# 06. API 비용 계산과 예상 비용 확인

## 비용을 확인하는 세 가지 시점

| 시점 | 방법 | 정확도 | 필요한 것 |
|---|---|---|---|
| **요청 전** (예상) | `count_tokens`로 입력 토큰 확인 + 출력 토큰 추정 → × 단가 | 입력은 정확, 출력은 추정 | 단가표 |
| **요청 후** (계산) | `response.usage` × 단가 | 거의 정확 | 단가표 |
| **청구 후** (실제) | Cost API (04 노트북) 또는 Console | 정확 | Admin Key |

앞의 두 방법은 **단가표**가 있어야 계산할 수 있다. 문제는 Anthropic API가 단가를 알려주지 않는다는 점이다. (01의 모델 목록에도 가격은 없음)

## 단가 출처: OpenRouter 공개 모델 API

[OpenRouter](https://openrouter.ai)는 여러 회사의 LLM을 한곳에서 쓸 수 있게 해주는 중개 서비스다. 이곳의 모델 목록 API는 **모델별 단가를 공개**하고 있다.

- `GET https://openrouter.ai/api/v1/models`: **API Key 없이 무료**로 조회
- 입력, 출력, 캐시 읽기/쓰기 단가와 구간별 단가까지 포함
- Anthropic 외 다른 회사 모델도 있어서, 여러 모델을 비교하는 프로젝트에 쓰기 좋음

### 주의할 점

- **공식 출처가 아니다.** OpenRouter가 표시하는 단가이지, Anthropic이 청구하는 금액의 근거는 아니다. 가격 변경이 늦게 반영되거나 틀릴 수 있다. 중요한 계산은 [공식 가격 페이지](https://platform.claude.com/docs/en/about-claude/pricing)와 한 번 대조한다.
- **모델 ID 형식이 다르다.** Anthropic은 `claude-opus-5-5`, OpenRouter는 `anthropic/claude-opus-5.5`라서 변환이 필요하다.
- 외부 서비스라 조회에 실패할 수 있다. 그래서 아래 코드는 실패 시 **예비 단가표**를 쓴다.
- fast mode, 데이터 지역 지정 등 특수 옵션의 추가 요금은 반영하지 않는다.

In [1]:
import re

import anthropic
import requests
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

MODEL = "claude-opus-5-5"

## 1. OpenRouter에서 단가표 받아오기

OpenRouter의 단가는 **토큰 1개당 달러(문자열)** 로 온다. 예: `"0.000004"` = 토큰당 $0.000004 = 100만 토큰당 $4

여기서는 계산하기 쉽게 **100만 토큰당 달러**로 바꿔서 저장한다.

| OpenRouter 필드 | 의미 |
|---|---|
| `prompt` | 입력 |
| `completion` | 출력 (thinking 포함) |
| `input_cache_read` | 캐시에서 읽은 입력 |
| `input_cache_write` | 캐시에 저장한 입력 (5분) |
| `input_cache_write_1h` | 캐시에 저장한 입력 (1시간) |
| `overrides` | 입력이 일정량(`min_prompt_tokens`)을 넘으면 적용되는 다른 단가. 입력에는 캐시 읽기/쓰기도 포함 |

In [2]:
OPENROUTER_MODELS_URL = "https://openrouter.ai/api/v1/models"

# 조회 실패 시 사용할 예비 단가표 (100만 토큰당 USD, 공식 가격 페이지 기준으로 직접 관리, 2026-10-18 확인)
FALLBACK_PRICING = {
    "claude-opus-5-5":   {"input": 4.00, "output": 20.00, "cache_read": 0.20, "cache_write_5m": 5.00,  "cache_write_1h": 8.00, "tiers": []},
    "claude-sonnet-5-5": {"input": 2.00, "output": 10.00, "cache_read": 0.10, "cache_write_5m": 2.50,  "cache_write_1h": 4.00, "tiers": []},
    "claude-haiku-5-5":  {"input": 0.10, "output": 0.50,  "cache_read": 0.01, "cache_write_5m": 0.125, "cache_write_1h": 0.20,
                          "tiers": [{"min_input_tokens": 100_000, "input": 0.50, "output": 2.50, "cache_read": 0.05, "cache_write_5m": 0.625, "cache_write_1h": 1.00}]},
}


def to_openrouter_id(model_id: str) -> str:
    """Anthropic 모델 ID → OpenRouter ID
    claude-opus-5-5 → anthropic/claude-opus-5.5
    claude-haiku-4-5-20251001 → anthropic/claude-haiku-4.5 (날짜 제거)
    claude-opus-5 → anthropic/claude-opus-5
    """
    base = re.sub(r"-\d{8}$", "", model_id)            # 끝의 날짜(8자리) 제거
    base = re.sub(r"-(\d+)-(\d+)$", r"-\1.\2", base)  # 마지막 버전 숫자 '5-5' → '5.5'
    return f"anthropic/{base}"


def _per_million(price: dict) -> dict:
    """OpenRouter 단가(토큰당, 문자열) → 100만 토큰당 float"""
    def get(key):
        return float(price.get(key, 0) or 0) * 1_000_000
    return {
        "input": get("prompt"),
        "output": get("completion"),
        "cache_read": get("input_cache_read"),
        "cache_write_5m": get("input_cache_write"),
        "cache_write_1h": get("input_cache_write_1h"),
    }


def load_pricing(model_ids: list[str]) -> tuple[dict, str]:
    """모델별 단가표와 출처를 반환. OpenRouter 조회 실패 시 예비 단가표 사용"""
    try:
        r = requests.get(OPENROUTER_MODELS_URL, timeout=30)
        r.raise_for_status()  # 200이 아니면 예외 발생
        by_id = {m["id"]: m["pricing"] for m in r.json()["data"]}
    except (requests.RequestException, ValueError, KeyError) as e:
        print(f"OpenRouter 조회 실패 → 예비 단가표 사용 ({e})")
        return dict(FALLBACK_PRICING), "fallback"

    pricing = {}
    for model_id in model_ids:
        price = by_id.get(to_openrouter_id(model_id))
        if price is None:
            continue  # OpenRouter에 없는 모델
        entry = _per_million(price)
        entry["tiers"] = [
            {"min_input_tokens": o["min_prompt_tokens"], **_per_million(o)}
            for o in price.get("overrides", [])
        ]
        pricing[model_id] = entry
    return pricing, "openrouter"


# 01에서 배운 모델 목록 조회로, 이 API Key로 쓸 수 있는 모델의 단가만 가져옴
available_models = [m.id for m in client.models.list()]
PRICING, PRICING_SOURCE = load_pricing(available_models)

print(f"단가 출처: {PRICING_SOURCE}\n")
print(f"{'Model':<28} {'Input':>8} {'Output':>8} {'CacheRead':>10} {'CacheWrite':>11}   (USD / 1M tokens)")
print("-" * 72)
for model_id in available_models:
    p = PRICING.get(model_id)
    if p is None:
        print(f"{model_id:<28} {'(단가 정보 없음)':>8}")
        continue
    tier_note = f"  * 입력 {p['tiers'][0]['min_input_tokens']:,} 토큰 초과 시 다른 단가" if p["tiers"] else ""
    print(f"{model_id:<28} {p['input']:>8.2f} {p['output']:>8.2f} {p['cache_read']:>10.2f} {p['cache_write_5m']:>11.3f}{tier_note}")

단가 출처: openrouter

Model                           Input   Output  CacheRead  CacheWrite   (USD / 1M tokens)
------------------------------------------------------------------------
claude-haiku-5-5                 0.10     0.50       0.01       0.125  * 입력 100,000 토큰 초과 시 다른 단가
claude-sonnet-5-5                2.00    10.00       0.10       2.500
claude-opus-5-5                  4.00    20.00       0.20       5.000
claude-fable-5-1                10.00    50.00       0.25      12.500
claude-opus-5                    5.00    25.00       0.50       6.250
claude-sonnet-5                  2.00    10.00       0.20       2.500
claude-fable-5                  10.00    50.00       1.00      12.500
claude-opus-4-8                  5.00    25.00       0.50       6.250
claude-opus-4-7                  5.00    25.00       0.50       6.250
claude-sonnet-4-6                3.00    15.00       0.30       3.750
claude-opus-4-6                  5.00    25.00       0.50       6.250
claude-opus-4-5-2025

## 2. 비용 계산 함수

03에서 배운 `usage` 필드를 단가와 곱한다. 캐시를 쓰면 입력이 세 종류로 나뉘고 단가도 각각 다르다.

```
비용 = 일반 입력       × 입력 단가
     + 캐시 읽기 입력  × 캐시 읽기 단가   (입력 단가의 1/10~1/40, 모델마다 다름)
     + 캐시 저장 입력  × 캐시 쓰기 단가   (입력 단가의 1.25배~2배)
     + 출력(thinking 포함) × 출력 단가
```

입력이 많아 다른 구간(`tiers`)에 해당하면 그 구간의 단가를 쓴다.

In [3]:
def get_rates(model_id: str, total_input_tokens: int) -> dict:
    """입력 토큰 수에 맞는 구간의 단가를 반환"""
    p = PRICING.get(model_id)
    if p is None:
        raise KeyError(f"단가표에 없는 모델: {model_id}")
    rates = p
    for tier in p["tiers"]:  # 기준을 '초과'하는 구간 중 가장 높은 구간 적용 (공식: "over 100,000 tokens")
        if total_input_tokens > tier["min_input_tokens"]:
            rates = tier
    return rates


def calc_cost(model_id: str, usage) -> dict:
    """response.usage로 비용(USD)을 항목별로 계산"""
    cache_read = usage.cache_read_input_tokens or 0
    cc = usage.cache_creation  # 캐시 저장 토큰의 5분/1시간 구분
    write_5m = (cc.ephemeral_5m_input_tokens if cc else usage.cache_creation_input_tokens) or 0
    write_1h = (cc.ephemeral_1h_input_tokens if cc else 0) or 0
    total_input = usage.input_tokens + cache_read + write_5m + write_1h

    r = get_rates(model_id, total_input)
    cost = {
        "input": usage.input_tokens * r["input"],
        "cache_read": cache_read * r["cache_read"],
        "cache_write": write_5m * r["cache_write_5m"] + write_1h * r["cache_write_1h"],
        "output": usage.output_tokens * r["output"],
    }
    cost = {k: v / 1_000_000 for k, v in cost.items()}  # 단가가 100만 토큰당이므로 나눔
    cost["total"] = sum(cost.values())
    return cost

## 3. 요청 전: 예상 비용 확인

요청을 보내기 전에 알 수 있는 것과 없는 것이 있다.

- **입력 토큰**: `count_tokens`(무료)로 **정확히** 알 수 있다.
- **출력 토큰**: 모델이 생성해 봐야 안다. 그래서 두 가지로 본다.
  - **예상치**: 원하는 답변 길이로 직접 추정한다. (한국어는 약 1글자 ≈ 1토큰, thinking은 별도로 더 붙음)
  - **최대치**: `max_tokens`만큼 다 쓴 경우. 비용이 이를 넘을 수 없다.

In [4]:
def estimate_cost(model_id: str, messages: list, expected_output_tokens: int, max_tokens: int = 16000, **kwargs) -> dict:
    """요청 전 예상 비용. 입력은 count_tokens로 정확히, 출력은 예상치/최대치로 계산"""
    count = client.messages.count_tokens(model=model_id, messages=messages, **kwargs)  # 무료
    r = get_rates(model_id, count.input_tokens)
    input_cost = count.input_tokens * r["input"] / 1_000_000
    return {
        "input_tokens": count.input_tokens,
        "expected": input_cost + expected_output_tokens * r["output"] / 1_000_000,
        "worst": input_cost + max_tokens * r["output"] / 1_000_000,
    }


messages = [{"role": "user", "content": "파이썬의 데코레이터가 무엇인지 예제 코드와 함께 설명해줘."}]

# 답변 1,000자 내외 + thinking 여유분 → 약 1,500 토큰으로 추정
est = estimate_cost(MODEL, messages, expected_output_tokens=1_500)

print(f"모델            : {MODEL}")
print(f"입력 토큰 (정확) : {est['input_tokens']:,}")
print(f"예상 비용        : ${est['expected']:.5f}   (출력 1,500 토큰 가정)")
print(f"최대 비용        : ${est['worst']:.5f}   (max_tokens 16,000 전부 사용 시)")

모델            : claude-opus-5-5
입력 토큰 (정확) : 43
예상 비용        : $0.03017   (출력 1,500 토큰 가정)
최대 비용        : $0.32017   (max_tokens 16,000 전부 사용 시)


### 같은 요청을 모델별로 비교 (무료)

모델을 고를 때 유용하다. 모델마다 **토크나이저(글자를 토큰으로 나누는 방식)가 달라** 같은 문장도 입력 토큰 수가 다를 수 있으므로, 모델별로 각각 센다.

In [5]:
compare_models = ["claude-opus-5-5", "claude-sonnet-5-5", "claude-haiku-5-5"]


def usd(x: float) -> str:
    return f"${x:.5f}"  # '$'를 숫자에 붙인 문자열로 만든 뒤 정렬해야 칸이 섞여 보이지 않음


print(f"{'Model':<20} {'Input tokens':>12} {'Expected':>12} {'Worst':>12}")
print("-" * 59)
for m in compare_models:
    e = estimate_cost(m, messages, expected_output_tokens=1_500)
    print(f"{m:<20} {e['input_tokens']:>12,} {usd(e['expected']):>12} {usd(e['worst']):>12}")

Model                Input tokens     Expected        Worst
-----------------------------------------------------------
claude-opus-5-5                43     $0.03017     $0.32017
claude-sonnet-5-5              43     $0.01509     $0.16009
claude-haiku-5-5               43     $0.00075     $0.00800


> 단가만 보면 Haiku가 훨씬 싸지만, 어려운 작업에서는 답변 품질이 다르다. 실제 모델 선택은 **비용과 품질을 함께** 비교해서 정한다.

## 4. 요청 후: 실제 사용량으로 비용 계산

> 이 셀은 메시지를 1회 보내므로 **소액의 요금이 발생한다.**

In [6]:
response = client.messages.create(model=MODEL, max_tokens=16000, messages=messages)

cost = calc_cost(response.model, response.usage)  # fallback 대비, 실제로 응답한 모델 기준
u = response.usage

print(f"입력 토큰  : {u.input_tokens:>6,}  → ${cost['input']:.5f}")
print(f"출력 토큰  : {u.output_tokens:>6,}  → ${cost['output']:.5f}   (thinking {u.output_tokens_details.thinking_tokens:,} 포함)")
print(f"캐시       :          → ${cost['cache_read'] + cost['cache_write']:.5f}")
print("-" * 40)
print(f"합계       :          → ${cost['total']:.5f}")

입력 토큰  :     43  → $0.00017
출력 토큰  :  3,065  → $0.06130   (thinking 280 포함)
캐시       :          → $0.00000
----------------------------------------
합계       :          → $0.06147


### 예상과 실제 비교

In [7]:
print(f"예상 출력 토큰 : {1_500:>6,}    실제 : {u.output_tokens:>6,}")
print(f"예상 비용      : ${est['expected']:.5f}   실제 : ${cost['total']:.5f}")
print(f"최대 비용      : ${est['worst']:.5f}   (실제 비용은 항상 이 이하)")

예상 출력 토큰 :  1,500    실제 :  3,065
예상 비용      : $0.03017   실제 : $0.06147
최대 비용      : $0.32017   (실제 비용은 항상 이 이하)


### 예상과 실제의 차이를 줄이려면

테스트 실행에서는 출력을 1,500 토큰으로 예상했지만 실제로는 3,600 토큰이 넘게 나왔다(예제 코드를 여러 개 넣어 길게 답함).

**오차는 전부 출력에서 생긴다.** 입력은 `count_tokens`로 정확히 알 수 있기 때문이다.

#### 차이를 0으로 만들 수는 없다

- 같은 프롬프트를 여러 번 보내도 **답변 길이가 매번 다르다.** 모델 출력에는 무작위성이 있다.
- thinking도 모델이 스스로 양을 정하므로(adaptive) 요청마다 달라진다.

따라서 목표는 "정확히 맞히기"가 아니라 **"차이를 줄이고, 얼마나 빗나갈지 범위로 말할 수 있게 하기"** 로 잡는 것이 현실적이다.

#### 방향 A. 출력을 예측 가능하게 만들기 (통제)

답변 길이 자체의 변동을 줄인다.
- 프롬프트에 길이를 명시한다 ("500자 이내로", "예제는 1개만")
- `effort`를 낮춰 thinking 양의 변동을 줄인다
- 구조화된 출력(정해진 JSON 형식)을 사용한다

#### 방향 B. 예측을 정교하게 만들기 (추정)

- **과거 기록 기반**: 04의 `log/usage_log.csv`에 작업 유형(요약, 설명, 코드 작성 등)을 함께 기록하고, 유형별·모델별 평균 출력 토큰을 예상치로 쓴다
- **점 대신 범위로**: "1,500 토큰"이 아니라 "보통 2,000, 90%의 경우 4,000 이하"처럼 **중앙값과 상위 90% 값**으로 예상한다. 예산을 잡을 때 더 쓸모 있다
- **입력과의 관계 활용**: 요약처럼 출력이 입력 길이에 비례하는 작업은 입력 토큰으로 출력을 추정할 수 있다
- 예산 초과가 절대 안 되는 경우에는 **최대 비용**(`max_tokens` 기준)을 기준으로 잡는다

#### 실험해 볼 수 있는 과제

1. 작업 유형 3~4개 × 프롬프트 몇 개를 준비한다
2. 같은 프롬프트를 **여러 번 반복** 요청해 출력 길이의 변동 폭을 측정한다
3. 길이 지시 유무, effort 단계별로 변동 폭을 비교한다
   - 예: "길이 지시가 있으면 편차가 얼마나 줄어드는가?", "모델이 '500자' 지시를 실제로 얼마나 지키는가?"
4. 평가 지표: 예상 대비 실제 **오차율**, 실제 값이 예상 범위 안에 들어온 **비율**

> **실험 비용**: 반복 요청이 많으므로 모델 선택이 중요하다. Opus 5.5로 50회 요청하면 대략 $2~4, Haiku 5.5로는 몇 센트 수준이다. 방법론 검증은 Haiku로 하고, 모델마다 답변 길이 성향이 다르므로 최종 결론은 실제로 사용할 모델로 확인한다.

## 정리

| 하고 싶은 것 | 방법 |
|---|---|
| 단가 확인 | OpenRouter 공개 API (무료, 비공식) → 공식 가격 페이지와 대조 |
| 요청 전 예상 비용 | `count_tokens`(무료) × 입력 단가 + 예상 출력 × 출력 단가 |
| 비용 상한 | `max_tokens` × 출력 단가 + 입력 비용 |
| 요청 후 비용 | `response.usage` × 단가 (`calc_cost`) |
| 실제 청구 금액 | 04 노트북의 Cost API (Admin Key) 또는 Console [Cost](https://platform.claude.com/cost) 페이지 |

**비용을 줄이는 방법** (뒤에서 다룰 주제)
- **프롬프트 캐싱**: 반복되는 앞부분을 캐시하면 그 부분의 입력 비용이 1/10 이하로 줄어든다.
- **Batch API**: 급하지 않은 대량 작업은 50% 할인
- **effort 조절**: 간단한 작업은 `low`로 thinking 토큰을 줄인다.

## 부록: 원화(KRW)로 보기

달러 금액을 원화로 바꿔서 본다. 환율은 **키 없이 무료로 쓸 수 있는 환율 API**에서 받아온다.

| 출처 | 주소 | 특징 |
|---|---|---|
| Frankfurter | `api.frankfurter.dev` | 유럽중앙은행(ECB) 기준환율. 평일 하루 1회 갱신 |
| ExchangeRate-API | `open.er-api.com` | 하루 1회 갱신. 첫 번째가 실패하면 사용 |
| 고정값 | 코드 안의 상수 | 둘 다 실패하면 사용 |

> 환율 전용 파이썬 라이브러리(`forex-python` 등)도 있지만, 내부적으로 쓰던 무료 API가 중단되어 동작하지 않는 경우가 많다. 그래서 여기서는 API를 `requests`로 직접 호출한다.

### 주의: 실제 청구 원화 금액과는 다르다

- 여기서 쓰는 환율은 **기준환율**(시장 중간값)이다. 실제로는 카드사 환율이 적용되고, **해외 결제 수수료**(보통 1% 내외)가 붙는다.
- 세금(부가세 등)이 별도로 붙을 수 있다.
- 따라서 원화 금액은 **"대략 이 정도"를 가늠하는 용도**로 쓴다. 정확한 금액은 카드 명세서나 Console의 청구 내역으로 확인한다.

In [8]:
# 두 API 모두 실패했을 때 쓰는 값 (2026-10-16 ECB 기준환율)
FALLBACK_USD_KRW = 1332.1


def get_usd_krw() -> tuple[float, str]:
    """1달러당 원화 환율과 출처를 반환. 차례대로 시도하고, 모두 실패하면 고정값 사용"""
    sources = [
        ("Frankfurter (ECB)", "https://api.frankfurter.dev/v1/latest?base=USD&symbols=KRW",
         lambda body: (body["rates"]["KRW"], body["date"])),
        ("ExchangeRate-API", "https://open.er-api.com/v6/latest/USD",
         lambda body: (body["rates"]["KRW"], body["time_last_update_utc"][:16])),
    ]
    for name, url, parse in sources:
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            rate, date = parse(r.json())
            return float(rate), f"{name}, {date}"
        except (requests.RequestException, ValueError, KeyError) as e:
            print(f"{name} 조회 실패 → 다음 출처 시도 ({e})")
    return FALLBACK_USD_KRW, "고정값 (2026-10-16)"


USD_KRW, RATE_SOURCE = get_usd_krw()
print(f"환율: 1 USD = {USD_KRW:,.2f} KRW  ({RATE_SOURCE})")


def krw(usd_amount: float) -> str:
    """달러 금액 → 원화 문자열. 1원 미만은 소수점 2자리까지 표시"""
    won = usd_amount * USD_KRW
    return f"{won:,.0f}원" if won >= 1 else f"{won:.2f}원"

환율: 1 USD = 1,332.10 KRW  (Frankfurter (ECB), 2026-10-16)


### 단가표를 원화로

100만 토큰 단위는 감이 잘 오지 않으므로 **1,000 토큰(한국어 약 1,000자)당** 금액도 함께 본다.

In [9]:
# 값 끝의 '원'이 화면에서 2칸을 차지하므로 머리글을 1칸씩 넓게 잡음
print(f"{'Model':<20} {'Input/1M':>13} {'Output/1M':>13} {'Input/1K':>11} {'Output/1K':>11}")
print("-" * 72)
for m in compare_models:
    p = PRICING[m]
    print(
        f"{m:<20} {krw(p['input']):>12} {krw(p['output']):>12} "
        f"{krw(p['input'] / 1000):>10} {krw(p['output'] / 1000):>10}"
    )

Model                     Input/1M     Output/1M    Input/1K   Output/1K
------------------------------------------------------------------------
claude-opus-5-5            5,328원      26,642원         5원        27원
claude-sonnet-5-5          2,664원      13,321원         3원        13원
claude-haiku-5-5             133원         666원      0.13원      0.67원


### 앞에서 계산한 비용을 원화로

3번(예상 비용)과 4번(실제 비용) 셀을 먼저 실행해야 한다.

In [10]:
# 한글은 화면에서 2칸을 차지해 f-string 칸 맞춤이 어긋나므로,
# 라벨은 같은 길이로 맞춰 두고 정렬하지 않음 ('예상 비용' = 화면 9칸)
rows = [
    ("예상 비용", est["expected"]),
    ("최대 비용", est["worst"]),
    ("실제 비용", cost["total"]),
]
print(f"{'':<9}  {'USD':>10}  {'KRW':>11}")  # KRW 값 끝의 '원'이 2칸이라 1칸 넓게
print("-" * 36)
for label, amount in rows:
    print(f"{label}  {usd(amount):>10}  {krw(amount):>10}")

# 같은 요청을 여러 번 보낸다면?
print()
for n in (100, 1_000):
    print(f"이 요청을 {n:,}번 보내면 약 {krw(cost['total'] * n)}")

                  USD          KRW
------------------------------------
예상 비용    $0.03017         40원
최대 비용    $0.32017        427원
실제 비용    $0.06147         82원

이 요청을 100번 보내면 약 8,189원
이 요청을 1,000번 보내면 약 81,887원


환율 정보가 자동으로 업데이트 되게 하려면..